## GLANCE_3_extract_patches

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import time
import ee
import datetime

In [2]:
# Authenticate ee. You first need to set up a free Google Cloud account
ee.Authenticate()

True

In [3]:
import geemap.foliumap as geemap  # Import Earth Engine and Geemap.foliumap
ee.Initialize()                   # Initialise Earth Engine

### Read GLANCE points for each biome

In [ ]:
# read geodataframe from file
gdf_biomeA_samples = gpd.read_parquet('../GLANCE_data/biomeA_glance.parquet') 
gdf_biomeB_samples = gpd.read_parquet('../GLANCE_data/biomeB_glance.parquet')      
gdf_biomeC_samples = gpd.read_parquet('../GLANCE_data/biomeC_glance.parquet')      
gdf_biomeD_samples = gpd.read_parquet('../GLANCE_data/biomeD_glance.parquet')    
gdf_biomeE_samples = gpd.read_parquet('../GLANCE_data/biomeE_glance.parquet')    
gdf_biomeF_samples = gpd.read_parquet('../GLANCE_data/biomeF_glance.parquet')  

#gdf = gpd.read_parquet('../GLANCE_data/biomeD_extracted.parquet')

In [ ]:
gdf = gdf_biomeB_samples.copy()
len(gdf)

In [ ]:
level3 = ['Water(0)','Developed(1)','Barren(2)','Trees(3)','Shrub(4)','Grassland(5)','Agriculture(6)'] 

# How many samples per landcover class

sum=0
for i in range(0,7):
    y=np.sum((gdf['level3'] == i))
    print(i, level3[i], '=', y)
    sum=sum+y
print('Total Samples =',sum,'\n')

## Read Random points for each biome

In [4]:
#####
region = 'biomeF'
year = '2021'
region, year

('biomeF', '2021')

In [5]:
# read geodataframe from file
filename = '../Inference_Points/' + region + '_random_points.parquet'
print(filename)

../Inference_Points/biomeF_random_points.parquet


In [6]:
gdf = gpd.read_parquet(filename)  
gdf.shape

(27623, 1)

In [7]:
# add lon and lat columns
gdf['lon'] = gdf.geometry.x
gdf['lat'] = gdf.geometry.y
gdf.shape

(27623, 3)

In [8]:
# remove the geometry column and add a row number column
gdf = gdf.drop(columns=['geometry'])
gdf.insert(loc=0, column='Sample', value=np.arange(len(gdf)))

In [9]:
gdf.head()

,Sample,lon,lat
0,0,-73.449608,9.954593
1,1,-72.440376,11.201045
2,2,-73.509316,10.049416
3,3,-71.493341,12.166031
4,4,-72.057612,11.605439


## Extract Training Data from Copernicus S2 Collection

In [ ]:
# for Glance points only
gdf_new = gdf.copy()
gdf.shape

In [ ]:
# extraction with check for sample_col size
print(datetime.datetime.now())
time_start = time.time()

missing_samples=[]

start_sample = 0
end_sample = len(gdf)

print('from',start_sample,'to',end_sample)

patch_all = np.zeros((0,10,5,5))            # sample, channels, height, width

for i in range(start_sample,end_sample):                               
    sample_no = gdf.iloc[i,0]
    lat = gdf.iloc[i,1]
    lon = gdf.iloc[i,2]
    end_year = str(gdf.iloc[i,4])
    start_date = end_year + '-01-01'
    end_date = end_year + '-12-31'

    point = ee.Geometry.Point(lon, lat)                         # ee.Geometry.Point
    
    collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')       # whole Sentinel-2 collection
        .filterDate(start_date, end_date)                       # time period - end year of GLANCE sample    
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 50))    # some images have no 'cloudy_pixel_percent' defined
        .filterBounds(point)                                    # only images interesecting this feature
    )

    num_images = collection.size().getInfo()     # no of images will be the same for 10m and 20m bands

    if (num_images == 0):
        print(i, ' sample:',sample_no, ' year:',end_year, ' images==0. no image will be added. remove gdf row')
        gdf_new = gdf_new.drop(gdf.index[i])
        missing_samples.append([sample_no,end_year])  
    else:
        coll_bands = collection.select(['B2','B3','B4','B8','B5','B6','B7','B8A','B11','B12']) # All bands of interest, 10m and 20m
        
        proj_10m = coll_bands.first().select('B2').projection()             # projection of 10m bands
        median_bands =  coll_bands.median().reproject(proj_10m)             # median is single multi-band image

        sample_col = median_bands.sample(region=point,scale=10,geometries=True)
        
        if sample_col.size().getInfo() == 0:
            print(i, ' sample:',sample_no, ' year:',end_year, ' no pixel found. remove gdf row')
            gdf_new = gdf_new.drop(gdf.index[i])
            missing_samples.append([sample_no,end_year])
        else:
            sample_fc = sample_col.first()

            lon_px, lat_px = sample_fc.getInfo()['geometry']['coordinates']

            point_px = ee.Geometry.Point(lon_px,lat_px)

            patch_geom = point_px.buffer(20).bounds()       # square box 40m x 40m centered on pixel

            np_image = geemap.ee_to_numpy(median_bands, region=patch_geom, scale=10)  # single image to numpy array [H x W x C]

            np_patch = np_image.transpose(2,0,1)                                          # to [C x H x W]
            np_patch = np.expand_dims(np_patch,0)                                         # to [B x C x H x W] = [1 x C x H x W]

            print(i, ' sample:',sample_no, 'num_images:', num_images)

            patch_all = np.concatenate((patch_all,np_patch),axis=0)

print('\n\npatch_all: ',patch_all.shape)

print('no of missing samples =', len(missing_samples))

time_end = time.time()
print('\n\nfinal run time=',(time_end-time_start)/60," minutes")


In [10]:
# extraction with no check for sample_col size
print(datetime.datetime.now())
time_start = time.time()

missing_samples=[]

start_sample = 21000
end_sample = len(gdf)
#end_sample = 21000

print('from',start_sample,'to',end_sample)

patch_all = np.zeros((0,10,5,5))            # sample, channels, height, width

for i in range(start_sample,end_sample):                               

    # GLANCE Data
    #sample_no = gdf.iloc[i,0]
    #lat = gdf.iloc[i,1]
    #lon = gdf.iloc[i,2]
    #end_year = str(gdf.iloc[i,4])

    # Random Points
    sample_no = gdf.iloc[i,0]
    lon = gdf.iloc[i,1]
    lat = gdf.iloc[i,2]
    end_year = year
    
    start_date = end_year + '-01-01'
    end_date = end_year + '-12-31'

    point = ee.Geometry.Point(lon, lat)                         # ee.Geometry.Point
    
    collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')       # whole Sentinel-2 collection
        .filterDate(start_date, end_date)                       # time period - end year of GLANCE sample    
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 50))    # some images have no 'cloudy_pixel_percent' defined
        .filterBounds(point)                                    # only images interesecting this feature
    )

    num_images = collection.size().getInfo()     # no of images will be the same for 10m and 20m bands

    if (num_images == 0):
        print(i, ' sample:',sample_no, ' year:',end_year, ' images==0. no image will be added. remove gdf row')
        gdf_new = gdf_new.drop(gdf.index[i])
        missing_samples.append([sample_no,end_year])  
    else:
        coll_bands = collection.select(['B2','B3','B4','B8','B5','B6','B7','B8A','B11','B12']) # All bands of interest, 10m and 20m
        
        proj_10m = coll_bands.first().select('B2').projection()             # projection of 10m bands
        median_bands =  coll_bands.median().reproject(proj_10m)             # median is single multi-band image

        sample_col = median_bands.sample(region=point,scale=10,geometries=True)
        
        sample_fc = sample_col.first()

        lon_px, lat_px = sample_fc.getInfo()['geometry']['coordinates']

        point_px = ee.Geometry.Point(lon_px,lat_px)

        patch_geom = point_px.buffer(20).bounds()       # square box 40m x 40m centered on pixel

        np_image = geemap.ee_to_numpy(median_bands, region=patch_geom, scale=10)  # single image to numpy array [H x W x C]

        np_patch = np_image.transpose(2,0,1)                                          # to [C x H x W]
        np_patch = np.expand_dims(np_patch,0)                                         # to [B x C x H x W] = [1 x C x H x W]

        print(i, ' sample:',sample_no, 'num_images:', num_images)

        patch_all = np.concatenate((patch_all,np_patch),axis=0)

print('\n\npatch_all: ',patch_all.shape)

print('no of missing samples =', len(missing_samples))

time_end = time.time()
print('\n\nfinal run time=',(time_end-time_start)/60," minutes")


2026-05-09 20:31:52.453443
from 21000 to 27623
21000  sample: 21000 num_images: 60
21001  sample: 21001 num_images: 63
21002  sample: 21002 num_images: 128
21003  sample: 21003 num_images: 116
21004  sample: 21004 num_images: 58
21005  sample: 21005 num_images: 58
21006  sample: 21006 num_images: 110
21007  sample: 21007 num_images: 52
21008  sample: 21008 num_images: 59
21009  sample: 21009 num_images: 69
21010  sample: 21010 num_images: 111
21011  sample: 21011 num_images: 136
21012  sample: 21012 num_images: 51
21013  sample: 21013 num_images: 61
21014  sample: 21014 num_images: 59
21015  sample: 21015 num_images: 247
21016  sample: 21016 num_images: 121
21017  sample: 21017 num_images: 65
21018  sample: 21018 num_images: 132
21019  sample: 21019 num_images: 125
21020  sample: 21020 num_images: 58
21021  sample: 21021 num_images: 67
21022  sample: 21022 num_images: 123
21023  sample: 21023 num_images: 63
21024  sample: 21024 num_images: 64
21025  sample: 21025 num_images: 59
21026  

## Save Patch Data to File - Glance Points

In [ ]:
# save patch data
np.save("../Patch_data/biomeB_8_patches.npy", patch_all)

# save label data
labels = gdf_new["level3"].to_numpy().astype(int)
np.save("../Patch_data/biomeB_8_patch_labels.npy", labels)

# save updated glance data
gdf_new.to_parquet('../Glance_data/biomeB_8_extracted.parquet')


## Save Patch Data to File - Random Points

In [11]:
file_name = '../Inference_Patches/biomeF4_random_patches_2021.npy'
file_name

'../Inference_Patches/biomeF4_random_patches_2021.npy'

In [12]:
# save patch data
np.save(file_name, patch_all)

## Load Data

In [ ]:
# load patch data
patch_loaded = np.load(file_name)
patch_loaded.shape

In [ ]:
# load patch labels
labels_loaded = np.load("../Patch_data/biomeB_8_patch_labels.npy")
labels_loaded.shape

In [ ]:
# load gdf
gdf_loaded = gpd.read_parquet('../Glance_data/biomeB_8_extracted.parquet')   
len(gdf_loaded)

In [13]:
# load random patch data
patches1 = np.load('../Inference_Patches/biomeF1_random_patches_2021.npy')
patches2 = np.load('../Inference_Patches/biomeF2_random_patches_2021.npy')
patches3 = np.load('../Inference_Patches/biomeF3_random_patches_2021.npy')
patches4 = np.load('../Inference_Patches/biomeF4_random_patches_2021.npy')

In [14]:
patches1.shape, patches2.shape, patches3.shape, patches4.shape

((7000, 10, 5, 5), (7000, 10, 5, 5), (7000, 10, 5, 5), (6623, 10, 5, 5))

In [15]:
patches_all = np.concatenate((patches1,patches2,patches3,patches4),axis=0)
patches_all.shape

(27623, 10, 5, 5)

In [16]:
np.save('../Inference_Patches/biomeF_random_patches_2021.npy', patches_all)

In [17]:
patches_load = np.load('../Inference_Patches/biomeF_random_patches_2021.npy')
patches_load.shape

(27623, 10, 5, 5)

## Plot a single Sentinel-2 image

In [ ]:
gdf_plot=gdf_new.copy()
len(gdf_plot)

In [ ]:
ind = 12000

In [ ]:
level3 = ['Water(0)','Developed(1)','Barren(2)','Trees(3)','Shrub(4)','Grassland(5)','Agriculture(6)']   

sample_no = gdf_plot.iloc[ind, 0]
lat = gdf_plot.iloc[ind, 1]
lon = gdf_plot.iloc[ind, 2]
end_year = str(gdf_plot.iloc[ind,4])
start_date = end_year + '-01-01'
end_date = end_year + '-12-31'

print('class3:',gdf_plot.iloc[ind,8],'class_label',level3[gdf_plot.iloc[ind,8]])
print('sample location: lon=',lon,' lat=',lat)

point = ee.Geometry.Point(lon, lat)                                                    # ee.Geometry.Point
box = ee.Geometry.BBox((lon-0.0004), (lat-0.0004), (lon+0.0004), (lat+0.0004))         # box is ee.Geometry.BBox
box_fc =  ee.FeatureCollection(box).style(**{'color':'red','fillColor': '#00000000'}) # convert to styled feature collection

collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')       # whole Sentinel-2 collection
    .filterDate(start_date, end_date)                       # time period - end year of GLANCE sample    
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 50))    # some images have no 'cloudy_pixel_percent' defined
    .filterBounds(point)                                    # only images interesecting this feature
)

num_images = collection.size().getInfo()     # no of images will be the same for 10m and 20m bands
print('num_images',num_images)

coll_bands = collection.select(['B2','B3','B4','B8','B5','B6','B7','B8A','B11','B12']) # All bands of interest, 10m and 20m
        
proj_10m = coll_bands.first().select('B2').projection()             # projection of 10m bands
median_bands =  coll_bands.median().reproject(proj_10m)             # median is single multi-band image

sample_fc = median_bands.sample(region=point, scale=10, geometries=True).first()  # point at centre of pixel lon/lat
lon_px, lat_px = sample_fc.getInfo()['geometry']['coordinates']
point_px = ee.Geometry.Point(lon_px,lat_px) 
    
patch_geom = point_px.buffer(20).bounds()       # square box 40m x 40m centered on pixel
     
vis = {'min':0.0, 'max':5000, 'bands': ['B4','B3','B2'],}           # RGB

Map = geemap.Map(height='600', width='60%')
Map.centerObject(point,zoom=20)
Map.addLayer(median_bands,vis,'median')  

# Add centre points. sample and pixel centre
Map.addLayer(point,{"color": "red"}, "point")
Map.addLayer(box_fc, {}, "box_fc")
Map.addLayer(point_px,{"color": "yellow"}, "point_px") 

# Add patch outline
patch_geom_fc =  ee.FeatureCollection(patch_geom).style(**{'color':'yellow','fillColor': '#00000000'}) 
Map.addLayer(patch_geom_fc, {}, "patch")

display(Map)


In [ ]:
sample_fc

## Plot Patch corresponding to Sentinel-2 image

In [ ]:
print('sample = ',ind,' level3 = ', level3[labels[ind]])

img1 = patch_all[ind]                        # select image number
img2 = img1.transpose(1,2,0)                     # C x H x W to H x W x C
img3 = img2[:,:,0:3]                             # BGR
img4 = img3[:,:,::-1]                            # RGB, reverse the order of the last dimension

print('img min=',img4.min())
print('img max=',img4.max())
img4 = img4/5000

img4 = img4

img4 = np.minimum(img4, 1)
img4 = np.maximum(img4, 0)

plt.figure(figsize=(5,5))
plt.imshow(img4)
plt.show()